# Experiment 7.2.1 — Phase-specific vs shared L2 readout

This frozen-checkpoint extension tests whether Exp7.2 L2 activity is already phase-aware, or whether downstream classification still benefits from an explicit phase-dependent weight matrix. No SNN is retrained.

Three probes are compared: **WholeCount shared** (128-D, one time-invariant weight matrix), **ordered Fixed250** (2048-D, phase-specific weights), and **phase-shuffled Fixed250** (2048-D matched-capacity control with absolute phase identity destroyed independently per sample).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'notebooks').exists():
    ROOT = ROOT.parent
ART = ROOT / 'notebooks' / 'artifacts' / 'experiment_7_2_1_phase_weight_probe' / 'phase_weight_probe_v1'
architectures = pd.read_csv(ART / 'architecture_table.csv')
summary = pd.read_csv(ART / 'probe_summary.csv')
deltas = pd.read_csv(ART / 'paired_deltas.csv')
ARCH_ORDER = architectures['architecture'].tolist()
SOURCE_ORDER = ['l2_wholecount_shared', 'l2_fixed250_phase_shuffled', 'l2_fixed250_ordered']
summary.shape, deltas.shape


## Primary test balanced accuracy

`l2_wholecount_shared` implements one shared temporal classifier because $W\sum_b z_b = \sum_b Wz_b$. `l2_fixed250_ordered` allows a distinct effective $W_b$ for each absolute 250-ms bin. The shuffled control keeps the same 2048-D capacity as ordered Fixed250 while destroying consistent phase labels.

In [ ]:
test_ba = summary[(summary['split'] == 'test')].copy()
table = test_ba.pivot_table(
    index='architecture',
    columns=['training_family', 'regularization', 'source'],
    values='balanced_accuracy_mean',
).reindex(ARCH_ORDER)
display((100 * table).round(2))


In [ ]:
families = ['local_tsce', 'e2e_wc']
regularizations = ['task_only', 'task_plus_reg']
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=True)
for ax, family, reg in zip(axes.ravel(), families * 2, ['task_only', 'task_only', 'task_plus_reg', 'task_plus_reg']):
    block = test_ba[(test_ba.training_family == family) & (test_ba.regularization == reg)]
    for source in SOURCE_ORDER:
        g = block[block.source == source].set_index('architecture').reindex(ARCH_ORDER)
        ax.errorbar(ARCH_ORDER, 100 * g['balanced_accuracy_mean'], yerr=100 * g['balanced_accuracy_std'], marker='o', label=source)
    ax.set_title(f'{family} / {reg}')
    ax.set_xlabel('Architecture')
    ax.set_ylabel('Test BA (%)')
    ax.tick_params(axis='x', rotation=30)
    ax.grid(alpha=0.25)
axes[0, 0].legend(fontsize=8)
fig.suptitle('Exp7.2.1 frozen L2 readouts')
fig.tight_layout()


## Paired attribution

The two primary contrasts are:

- **ordered − wholecount**: benefit of allowing phase-specific weights instead of one shared temporal weight matrix.
- **ordered − phase-shuffled**: benefit of consistent absolute phase alignment at matched 2048-D feature capacity.

If both are positive, the cleanest interpretation is that useful phase information remains external to L2 and must still modulate the downstream evidence weighting.

In [ ]:
ba_delta = deltas[deltas['metric'] == 'balanced_accuracy'].copy()
primary = ba_delta[ba_delta['contrast'].isin(['ordered_minus_wholecount', 'ordered_minus_phase_shuffled'])]
delta_table = primary.pivot_table(
    index='architecture',
    columns=['training_family', 'regularization', 'contrast'],
    values='delta_mean',
).reindex(ARCH_ORDER)
display((100 * delta_table).round(2))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharey=True)
for ax, family, reg in zip(axes.ravel(), families * 2, ['task_only', 'task_only', 'task_plus_reg', 'task_plus_reg']):
    block = primary[(primary.training_family == family) & (primary.regularization == reg)]
    for contrast in ['ordered_minus_wholecount', 'ordered_minus_phase_shuffled']:
        g = block[block.contrast == contrast].set_index('architecture').reindex(ARCH_ORDER)
        ax.errorbar(ARCH_ORDER, 100 * g['delta_mean'], yerr=100 * g['delta_std'], marker='o', label=contrast)
    ax.axhline(0, linewidth=1)
    ax.set_title(f'{family} / {reg}')
    ax.set_xlabel('Architecture')
    ax.set_ylabel('Paired BA gain (pp)')
    ax.tick_params(axis='x', rotation=30)
    ax.grid(alpha=0.25)
axes[0, 0].legend(fontsize=8)
fig.suptitle('Does L2 still require explicit phase-conditioned weights?')
fig.tight_layout()


In [ ]:
ranked = primary.sort_values('delta_mean', ascending=False)[
    ['contrast', 'architecture', 'training_family', 'regularization', 'delta_mean', 'delta_std', 'n']
].copy()
ranked['delta_mean_pp'] = 100 * ranked['delta_mean']
ranked['delta_std_pp'] = 100 * ranked['delta_std']
display(ranked.drop(columns=['delta_mean', 'delta_std']).head(30).round(2))
